In [1]:
import os
import time
import requests
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

In [2]:
### LLM Initialization
### Prompt Generation
### Sample Responses

### Document Reading
### Embedding Module Initialization
### FAISS DB Initialization
### Chain Initialization

### Evaluation

### LLM

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import pipeline
from langchain import HuggingFacePipeline

In [4]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, temperature=0.01, do_sample=True)
llm = HuggingFacePipeline(pipeline=pipe)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

C:\Users\citak\AppData\Local\Temp\ipykernel_14968\3825940560.py:12: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  llm = HuggingFacePipeline(pipeline=pipe)


In [5]:
prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=128
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response)

c:\Users\citak\anaconda3\envs\pytorch\lib\site-packages\transformers\models\qwen2\modeling_qwen2.py:544: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Certainly! A large language model (LLM) is a type of artificial intelligence designed to understand and generate human-like text based on the patterns it has learned from vast amounts of data. These models can be trained on a wide variety of texts, including books, articles, web pages, and more.

Key features of LLMs include:

1. **Size and Complexity**: They are often highly complex, with millions or even billions of parameters, which allows them to capture intricate patterns in language.

2. **Training Data**: They are typically trained using unsupervised learning techniques, meaning they learn from unlabelled text data. This enables


### Prompt Generation

In [6]:
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain.schema.messages import SystemMessage, HumanMessage, AIMessage

In [23]:
template = """
You are an intelligent assistant that answers questions based on the given context.
If the answer is not found in the context, just say "I don't know" else return your answer in Json block

Context:
{context}

Question:
{question}

```json```
"""

In [24]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

formatted_prompt = prompt.format(
    context = "Galatasaray and Fenerbahce are the two biggest clubs of Turkish National Football League.",
    question = "Which country does Galatasaray play in?"
)

response = llm.invoke(formatted_prompt)

print(response)


You are an intelligent assistant that answers questions based on the given context.
If the answer is not found in the context, just say "I don't know" else return your answer in Json block

Context:
Galatasaray and Fenerbahce are the two biggest clubs of Turkish National Football League.

Question:
Which country does Galatasaray play in?

```json```
{
  "answer": "Turkey"
}
```json``` I don't know
The context mentions that Galatasaray is a club from the Turkish National Football League, but it does not explicitly state which country they play in. Given the information provided, we can infer that Galatasaray plays in Turkey. However, since the direct statement is missing, the most accurate response based solely on the given context is "I don't know."


### Document Reading

In [9]:
import pypdf
from langchain_community.document_loaders import PyPDFLoader

In [10]:
pdf_files = ['data/Business-Conduct-Policy.pdf', 'data/third-party-code.pdf']
pages = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    async for page in loader.alazy_load():
        pages.append(page)

Ignoring wrong pointing object 13 0 (offset 0)


In [11]:
len(pages), pages[-4]

(28,
 Document(metadata={'source': 'data/third-party-code.pdf', 'page': 4}, page_content='dining, food preparation, and storage facilities provided to employees must be \nsanitary. \nHealth and Safety Communication. Third parties must provide workers with \nappropriate workplace health and safety training in their primary language. \nHealth and safety related information shall be clearly posted in the facilities. \nLaw and Ethics  \nApple expects the highest standards of ethical conduct in all of our endeavors. \nWe expect our third parties to comply with all applicable laws and and be ethical \nin every aspect of their business.  \nCorruption. Third parties may not engage in corruption, extortion, \nembezzlement, or bribery. A bribe is defined as offering or receiving anything of \nvalue to any person for the purpose of obtaining or retaining business, or \nsecuring an improper advantage. Kickbacks are a type of bribery and occur when \na person is offered money or something of value 

### Embedding Module Initialization

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

In [33]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [34]:
sample_embedding = embeddings.embed_query(pages[0].page_content)
print(np.array(sample_embedding).shape)

(384,)


### FAISS DB Initialization

In [35]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_text_splitters import RecursiveCharacterTextSplitter


from uuid import uuid4

In [45]:
## Add documents to the vector store
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(pages)

In [46]:
vector_db = FAISS.from_documents(all_splits, embeddings)
vector_db_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

### Chain Initialization

In [39]:
from langchain.chains import RetrievalQA

In [47]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db_retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

### Inference

In [48]:
question = "What are the ethical rules of Apple?"
result = qa_chain.invoke({"query": question})


print("\nAnswer:", result['result'])
print("\n--- Retrieved Docs ---")
for i, doc in enumerate(result["source_documents"]):
    print(f"[{i+1}] {doc.page_content[:200]}...\n")


Answer: 
You are an intelligent assistant that answers questions based on the given context.
If the answer is not found in the context, just say "I don't know" else return your answer in Json block

Context:
3
Business Conduct February 2025
Introduction Behaviors Protecting Apple Accountability Integrity Resources
The following principles guide Apple’s business practices:
•  Honesty—Demonstrate honesty and high ethical standards in all business dealings.
•  Respect—Treat customers, partners, suppliers, employees, and others with respect and courtesy. 
• Confidentiality—Protect Apple Confidential Information, including that of our customers, partners, and suppliers.
• Compliance—Ensure that business decisions comply with applicable laws and regulations.
Apple expects its suppliers, contractors, consultants, and other business partners to follow these principles when providing 
goods and services to Apple or acting on our behalf. Third parties involved in manufacturing, components, sour